In [ ]:
import wikipediaapi
import os
import urllib.parse
import random
import time
import requests
import urllib.parse

In [ ]:
categories_links_dict = {}

categories_dict = {
    "История, Краеведение, Политика": [
        "Древний_мир",
        "Международные_отношения",
        "Дипломатия",
        "Политические_партии_по_алфавиту",
        "Государственное_управление",
        "Войны_XX_века",
        "Политология",
        "Холодная_война",
        "Археологические_культуры_по_алфавиту",
        "Дворянские_роды_России",
        "Революционеры_России",
        "Исторические_хронологии",
        "Политические_теории",
        "Государство",
        "Всемирная_история"
    ],

    "Культура, Искусство": [
        "Картины_по_алфавиту",
        "Синглы_по_алфавиту",
        "Музыкальные_фестивали_России",
        "Музеи_по_алфавиту",
        "Изобразительное_искусство",
        "Массовая_культура"
    ],

    "Общество, Экономика и Право": [
        "Макроэкономика",
        "Банковское_дело",
        "Русский_язык",
        "Демография",
        "Федеральные_законы_Российской_Федерации",
        "Философия",
        "Экономика",
        "Нумизматика",
        "Этнография",
        "Археология",
        "Языкознание"
    ],

    "Игры, развлечения": [
        "Компьютерные_игры_по_алфавиту",
        "Настольные_игры_по_алфавиту",
        "Головоломки",
        "Парки_развлечений",
        "Игрушки",
        "Косплей",
        "Киберспорт",
        "Коллекционирование",
        "Фестивали_по_алфавиту",
        "Персонажи_аниме_и_манги",
        "Дополнения_к_компьютерным_играм",
        "Персонажи_компьютерных_игр"
    ],

    "Наука и Технологии": [
        "Информационные_технологии",
        "Искусственный_интеллект",
        "Робототехника",
        "Электроника",
        "Философские_теории",
        "Историография",
        "Растения_по_алфавиту",
        "Генетика",
        "Гены",
        "Ферменты",
        "Анатомия_человека",
        "Эндемики_России",
        "Научные_дисциплины",
        "История_науки",
        "Научные_исследования",
        "Научные_понятия",
        "Физические_константы",
        "Математические_теоремы",
        "Лекарственные_средства",
        "Космические_аппараты"
    ],
}

In [ ]:
def get_links_from_wiki(category_name):
    url = "https://ru.wikipedia.org/w/api.php"
    headers = {
        'User-Agent': 'MY_USER_AGENT'
    }
    links = []
    cmcontinue = None

    # Список букв для случайного старта поиска (чтобы не всегда начинать с 'А')
    alphabet = "АБВГДЕЖЗИКЛМНОПРСТУФХЦЧШЩЭЮЯ"
    random_start = random.choice(alphabet)

    while True:
        params = {
            "action": "query",
            "list": "categorymembers",
            "cmtitle": f"Category:{category_name}",
            "cmlimit": 500,
            "cmtype": "page",
            "format": "json",
            "cmcontinue": cmcontinue,
            "cmstartsortkeyprefix": random_start
        }

        try:
            response = requests.get(url, params=params, headers=headers, timeout=30)

            if response.status_code != 200:
                print(f"   [!] Ошибка сервера: {response.status_code}")
                break

            data = response.json()

            if 'query' in data and 'categorymembers' in data['query']:
                pages = data['query']['categorymembers']
                links.extend([f"https://ru.wikipedia.org/wiki/{p['title'].replace(' ', '_')}" for p in pages])

            cmcontinue = data.get('continue', {}).get('cmcontinue')

            if not cmcontinue or len(links) >= 1000:
                break

            time.sleep(0.5)

        except Exception as e:
            print(f"Ошибка в {category_name}: {e}")
            break

    random.shuffle(links)

    return links[:300]


In [ ]:
#Парсим текст
categories_links_dict = {category: [] for category in categories_dict.keys()}

for list_categories in categories_dict.keys():
    print(f"Обрабатываем список {list_categories, list(categories_dict.keys()).index(list_categories)}")
    for category in categories_dict[list_categories]:
        time.sleep(0.5)
        links = get_links_from_wiki(category)
        categories_links_dict[list_categories].extend(links)


In [ ]:
#Создаем txt с текстом
wiki_html = wikipediaapi.Wikipedia(
    user_agent='MyHallucinationDetectorBot/1.0 (contact: aleekseev.max@google.com; purpose: research)',
    language='ru',
    extract_format=wikipediaapi.ExtractFormat.WIKI
)

In [ ]:
stops = [232, 265]
num = -1

for categories_links in list(categories_links_dict.keys())[3:]:
    num += 1
    print(f"Обрабатываем список {categories_links} (Индекс: {num})")

    save_path = f"PATH_PATH_PATH/{categories_links}"
    os.makedirs(save_path, exist_ok=True)

    counter = 0
    for link in categories_links_dict[categories_links]:
        page_name_raw = link.split("/wiki/")[-1]
        page_name = urllib.parse.unquote(page_name_raw).replace('_', ' ')

        time.sleep(0.25)
        page = wiki_html.page(page_name)

        if page.exists():
            counter += 1
            filename = f'{categories_links[:3]}_{counter}_wiki.txt'
            full_path = os.path.join(save_path, filename)

            with open(full_path, 'a', encoding='utf-8') as file:
                file.write(page.title + " " + page.text)

            if counter >= stops[num]:
                print(f"   Достигнут лимит {stops[num]} для этой категории.")
                break
        else:
            print(f"   [!] Страница '{page_name}' не найдена.")

print("Все категории обработаны")


In [ ]:
for key in categories_links_dict.keys():
    print("____________________")
    print(len(categories_links_dict[key]))